# Agentic RAG Assistant
Kaggle setup: turn on Internet, choose GPU T4 x2, and add the 11 data files as a Kaggle dataset.

## 1. Libraries

In [2]:
!pip install -q chromadb rank_bm25 pdfplumber python-docx bitsandbytes accelerate sentence-transformers gradio

import os
import re
import ast
import glob
import operator
import warnings
import numpy as np
import pandas as pd
import torch
import pdfplumber
import chromadb
import gradio as gr
from docx import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

warnings.filterwarnings("ignore")
print("GPU available:", torch.cuda.is_available())

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
GPU available: True


## 2. Settings

In [23]:
data_folder = "/kaggle/input"
llm_name = "Qwen/Qwen2.5-7B-Instruct"
embed_name = "BAAI/bge-large-en-v1.5"
rerank_name = "BAAI/bge-reranker-large"

chunk_max_chars = 1000
top_dense = 15
top_bm25 = 15
top_rerank_input = 12
top_final = 5
min_rerank_score = 0.005
max_steps = 7

# the company data is written as of this date
today = pd.Timestamp("2026-11-24")

## 3. Read the files

In [7]:
files = [p for p in glob.glob(data_folder + "/**/*", recursive=True) if os.path.isfile(p) and p.lower().endswith((".pdf", ".docx", ".txt", ".csv"))]
print(len(files), "files found")
for p in files:
    print(p)

heading_pattern = re.compile(r"^(\d+)\.\s+[A-Z][^.\n]{0,60}$")

def split_numbered(text):
    sections = []
    title = "General"
    body = []
    expected = 1
    for line in text.split("\n"):
        line = line.strip()
        match = heading_pattern.match(line)
        if match and int(match.group(1)) == expected:
            sections.append((title, "\n".join(body)))
            title = line
            body = []
            expected += 1
        elif line != "":
            body.append(line)
    sections.append((title, "\n".join(body)))
    return sections

def split_faq(text):
    blocks = text.split("\nQ: ")
    sections = [("General", blocks[0].strip())]
    for block in blocks[1:]:
        question, answer = block.split("\nA: ", 1)
        sections.append((question.strip(), "Q: " + question.strip() + "\nA: " + answer.strip()))
    return sections

def split_by_blank_heading(text):
    lines = [l.strip() for l in text.split("\n")]
    sections = []
    title = "General"
    body = []
    for i, line in enumerate(lines):
        is_heading = i > 0 and line != "" and len(line) < 40 and ":" not in line and not line.endswith(".") and not line.startswith("-") and lines[i - 1] == "" and i + 1 < len(lines) and lines[i + 1] != ""
        if is_heading:
            sections.append((title, "\n".join(body)))
            title = line
            body = []
        elif line != "":
            body.append(line)
    sections.append((title, "\n".join(body)))
    return sections

def read_docx(path):
    doc = Document(path)
    sections = []
    title = "General"
    body = []
    for p in doc.paragraphs:
        text = p.text.strip()
        if text == "":
            continue
        if p.style.name.startswith("Heading"):
            sections.append((title, "\n".join(body)))
            title = text
            body = []
        else:
            body.append(text)
    sections.append((title, "\n".join(body)))
    return sections

def read_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return split_numbered(text)

def read_txt(path):
    text = open(path, encoding="utf-8").read()
    if "\nQ: " in text:
        return split_faq(text)
    if any(heading_pattern.match(l.strip()) for l in text.split("\n")):
        return split_numbered(text)
    return split_by_blank_heading(text)

def read_csv_rows(path):
    table = pd.read_csv(path)
    sections = []
    for i, row in table.iterrows():
        parts = [c.replace("_", " ") + ": " + str(row[c]) for c in table.columns]
        sections.append(("row " + str(i + 1) + " (" + str(row.iloc[0]) + ")", ", ".join(parts)))
    return sections

11 files found
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Customer_contract.pdf
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Employee_handbook.docx
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Employees.csv
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/HR_policy.pdf
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/IT_security_policy.txt
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Incident_report.txt
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Offices_directory.csv
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Pricing.csv
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Product_guide.pdf
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Sales_data.csv
/kaggle/input/datasets/shrouqmansy/novatech-docs-and-data/novatech_data/Support_FA

## 4. Clean and split into chunks

In [8]:
def cut_text(text):
    if len(text) <= chunk_max_chars:
        return [text]
    sentences = re.split(r"(?<=[.!?])\s+", text)
    pieces = []
    current = []
    size = 0
    for s in sentences:
        if size + len(s) > chunk_max_chars and current:
            pieces.append(" ".join(current))
            current = [current[-1]]
            size = len(current[0])
        current.append(s)
        size += len(s)
    pieces.append(" ".join(current))
    return pieces

all_chunks = []
for path in files:
    name = os.path.basename(path)
    ext = name.lower().rsplit(".", 1)[1]
    if ext == "docx":
        sections = read_docx(path)
    elif ext == "pdf":
        sections = read_pdf(path)
    elif ext == "txt":
        sections = read_txt(path)
    else:
        sections = read_csv_rows(path)
    for title, body in sections:
        body = re.sub(r"[ \t]+", " ", body).strip()
        if len(body) < 20:
            continue
        for piece in cut_text(body):
            text = "Document: " + name + " | Section: " + title + "\n" + piece
            all_chunks.append({"id": "c" + str(len(all_chunks)), "text": text, "source": name + " / " + title, "file": name})

chunk_by_id = {c["id"]: c for c in all_chunks}
print(len(all_chunks), "chunks created")
print(pd.Series([c["file"] for c in all_chunks]).value_counts())
print()
print(all_chunks[5]["text"])

113 chunks created
Sales_data.csv            24
Employees.csv             16
Support_FAQ.txt           11
IT_security_policy.txt    10
Employee_handbook.docx     9
Product_guide.pdf          9
Customer_contract.pdf      8
HR_policy.pdf              8
Offices_directory.csv      8
Incident_report.txt        7
Pricing.csv                3
Name: count, dtype: int64

Document: Customer_contract.pdf | Section: 5. Data
Closed tickets are retained for 36 months. Customer data is hosted in Frankfurt, Germany. After
termination, data is kept for 30 days and then permanently deleted.


## 5. Embeddings and vector database

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(embed_name, device=device)
chunk_vecs = embedder.encode([c["text"] for c in all_chunks], normalize_embeddings=True, batch_size=16, show_progress_bar=False)

chroma_client = chromadb.PersistentClient(path="/kaggle/working/chroma_db")
try:
    chroma_client.delete_collection("novatech")
except Exception:
    pass
collection = chroma_client.create_collection(name="novatech", metadata={"hnsw:space": "cosine"})
collection.add(ids=[c["id"] for c in all_chunks], documents=[c["text"] for c in all_chunks], embeddings=chunk_vecs.tolist(), metadatas=[{"source": c["source"], "file": c["file"]} for c in all_chunks])

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in all_chunks])
print("stored in chroma:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

stored in chroma: 113


## 6. Retrieval (hybrid search + reranker)

In [10]:
reranker = CrossEncoder(rerank_name, max_length=512, device=device)

def retrieve(query):
    query_vec = embedder.encode(["Represent this sentence for searching relevant passages: " + query], normalize_embeddings=True)
    dense = collection.query(query_embeddings=query_vec.tolist(), n_results=top_dense)
    dense_ids = dense["ids"][0]

    bm25_scores = bm25.get_scores(tokenize(query))
    best = np.argsort(bm25_scores)[::-1][:top_bm25]
    bm25_ids = [all_chunks[i]["id"] for i in best if bm25_scores[i] > 0]

    fused = {}
    for rank, cid in enumerate(dense_ids):
        fused[cid] = fused.get(cid, 0) + 1 / (60 + rank)
    for rank, cid in enumerate(bm25_ids):
        fused[cid] = fused.get(cid, 0) + 1 / (60 + rank)
    candidate_ids = sorted(fused, key=fused.get, reverse=True)[:top_rerank_input]

    pairs = [[query, chunk_by_id[cid]["text"]] for cid in candidate_ids]
    scores = np.array(reranker.predict(pairs))
    # some versions return raw scores, so turn them into 0-1 values
    if scores.min() < 0 or scores.max() > 1:
        scores = 1 / (1 + np.exp(-scores))

    results = []
    for i in np.argsort(scores)[::-1][:top_final]:
        if scores[i] >= min_rerank_score:
            chunk = chunk_by_id[candidate_ids[i]]
            results.append({"id": chunk["id"], "text": chunk["text"], "source": chunk["source"], "score": float(scores[i])})
    return results

for r in retrieve("how many days of annual leave"):
    print(round(r["score"], 3), r["source"])

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

0.984 HR_policy.pdf / 3. Annual leave
0.329 HR_policy.pdf / 4. Sick leave
0.061 HR_policy.pdf / 5. Maternity and paternity leave


## 7. Language model

In [11]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name, quantization_config=bnb_config, device_map="auto")

def ask_llm(messages, max_new_tokens=400):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(ask_llm([{"role": "user", "content": "Say hello in one short sentence."}], 30))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Hello!


## 8. Tools

In [12]:
def find_file(name):
    for p in files:
        if os.path.basename(p).lower() == name.lower():
            return p
    return None

sales = pd.read_csv(find_file("Sales_data.csv"), parse_dates=["Date"])
employees = pd.read_csv(find_file("Employees.csv"), parse_dates=["Hire_date"])
pricing = pd.read_csv(find_file("Pricing.csv"))
offices = pd.read_csv(find_file("Offices_directory.csv"))

table_info = "sales columns: " + ", ".join(sales.columns) + "\nemployees columns: " + ", ".join(employees.columns) + "\npricing columns: " + ", ".join(pricing.columns) + "\noffices columns: " + ", ".join(offices.columns)

evidence = []

def add_evidence(text, label):
    for i, e in enumerate(evidence):
        if e["text"] == text:
            return i + 1
    evidence.append({"text": text, "label": label})
    return len(evidence)

def search_docs(query):
    found = retrieve(query)
    if len(found) == 0:
        return "No relevant passages found."
    lines = []
    for item in found:
        add_evidence(item["text"], item["source"])
        lines.append(item["text"])
    return "\n\n".join(lines)

def summarize(target):
    key = target.strip().lower().replace(" ", "_")
    picked = [c for c in all_chunks if c["file"].lower() == key or c["file"].lower().rsplit(".", 1)[0] == key]
    if len(picked) == 0:
        picked = retrieve(target)
    if len(picked) == 0:
        return "No relevant passages found."
    text = "\n\n".join(c["text"] for c in picked)[:6000]
    messages = [{"role": "system", "content": "Summarize the text in 3 to 5 short bullet points. Keep every number, date and name exactly as written. Use only the text."}, {"role": "user", "content": text}]
    summary = ask_llm(messages, 300)
    add_evidence("Summary of " + target + ":\n" + summary, "Summary of " + target)
    return summary

def analyze_data(expression):
    expression = expression.strip().strip("`").strip()
    banned = ["__", "import", "open(", "exec", "eval", "os.", "sys.", "to_csv", "to_pickle"]
    if any(b in expression for b in banned):
        return "Error: this expression is not allowed."
    names = {"sales": sales, "employees": employees, "pricing": pricing, "offices": offices, "pd": pd, "today": today}
    safe = {"len": len, "sum": sum, "round": round, "min": min, "max": max, "sorted": sorted, "list": list, "str": str, "int": int, "float": float, "abs": abs}
    try:
        result = eval(expression, {"__builtins__": safe}, names)
    except Exception as e:
        return "Error: " + str(e)
    if isinstance(result, (pd.DataFrame, pd.Series)):
        result = result.to_string()
    result = str(result)[:1500]
    add_evidence("Table query: " + expression + "\nResult:\n" + result, "Data tables")
    return result

math_ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}

def eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in math_ops:
        return math_ops[type(node.op)](eval_node(node.left), eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in math_ops:
        return math_ops[type(node.op)](eval_node(node.operand))
    raise ValueError("not a simple math expression")

def calculator(expression):
    expression = expression.strip().strip("`").replace(",", "").replace("^", "**")
    try:
        result = round(eval_node(ast.parse(expression, mode="eval").body), 4)
    except Exception as e:
        return "Error: " + str(e)
    add_evidence("Calculation: " + expression + " = " + str(result), "Calculator")
    return str(result)

tools = {"search_docs": search_docs, "summarize": summarize, "analyze_data": analyze_data, "calculator": calculator}

print(calculator("12 * 39 * 12 * 0.8"))
print(analyze_data('sales[sales["Plan"] == "Business"]["Amount_USD"].sum()'))

4492.8
21138.0


In [37]:
def date_math(text):
    match = re.search(r"(\d{4}-\d{2}-\d{2})\s*([+-])\s*(\d+)", text)
    if match is None:
        return "Error: write it like 2027-02-01 - 7"
    days = int(match.group(3))
    found_rule = any(re.search(r"\b" + str(days) + r"\s+(calendar\s+|working\s+|business\s+)?days", e["text"]) for e in evidence if e["label"] != "Date calculation")
    if not found_rule:
        return "Error: " + str(days) + " days was not found in the documents. Search the documents first for the rule, for example cancellation notice before renewal, and use the number of days you find."
    start = pd.Timestamp(match.group(1))
    if match.group(2) == "-":
        result = start - pd.Timedelta(days=days)
    else:
        result = start + pd.Timedelta(days=days)
    result = result.strftime("%B %d, %Y")
    add_evidence("Date calculation: " + text.strip() + " = " + result, "Date calculation")
    return result

tools["date_math"] = date_math
print(date_math("2027-02-01 - 7 days"))

Error: 7 days was not found in the documents. Search the documents first for the rule, for example cancellation notice before renewal, and use the number of days you find.


In [33]:
def search_docs(query):
    found = retrieve(query)
    if len(found) == 0:
        rewrite_messages = [{"role": "system", "content": "Rewrite the search query as 3 to 6 plain keywords. Return only the keywords."}, {"role": "user", "content": query}]
        short_query = ask_llm(rewrite_messages, 20)
        found = retrieve(short_query)
    if len(found) == 0:
        return "No relevant passages found. Try a shorter query with 3 to 5 keywords."
    lines = []
    for item in found:
        add_evidence(item["text"], item["source"])
        lines.append(item["text"])
    return "\n\n".join(lines)

tools["search_docs"] = search_docs

for r in retrieve("cancellation notice period from pricing table or relevant documents"):
    print(round(r["score"], 3), r["source"])

## 9. The agent

In [39]:
plan_prompt = """You plan how to answer a question about NovaTech company documents and data tables.
Write the smallest number of simple steps needed, from 1 to 4 numbered lines. Do not answer the question.
Never skip a step that finds a rule or a number you need, such as notice days, limits or discounts.
A simple question needs only one step.
Example question: What is the monthly fee of customer X and when must they cancel?
Example plan:
1. Find the plan and monthly fee of customer X in the sales table
2. Search the contract for the term end date
3. Search the documents for the cancellation notice days
4. Calculate the cancellation date
Tables available:
""" + table_info

agent_prompt = """You are a careful assistant for NovaTech Solutions. You answer questions only by using tools.

Tools:
search_docs: search the company documents (policies, handbook, product guide, FAQ, contract, incident report). Input: a short search query of 3 to 5 keywords.
summarize: summarize one document or topic. Input: a file name like Incident_report.txt, or a topic.
analyze_data: run ONE pandas expression on the tables below. Input: a pandas expression.
calculator: do math. Input: an arithmetic expression such as 12 * 39 * 0.8
date_math: add or subtract days from a date. Input: a date and a number of days, such as 2031-03-15 - 4

Tables:
""" + table_info + """
Plan values: Starter, Business, Enterprise. Billing values: Monthly, Annual. The variable today holds the current date.
Example: sales[sales["Plan"] == "Business"]["Amount_USD"].sum()

Use this format exactly:
Thought: what you need to do next
Action: one of search_docs, summarize, analyze_data, calculator, date_math
Action Input: the input for the tool

You will then see an Observation. Repeat until you have all the facts you need, then write:
Thought: I have enough information
Final Answer: done

Rules:
- Search each part of a complex question separately with short keyword queries.
- Search company rules with topic words only, without customer or employee names. For example search cancellation notice before renewal, not Delta Logistics cancellation.
- Facts about a customer or an employee come from the tables or from the contract and incident report.
- If a search finds nothing useful, try again with fewer and different keywords.
- Never assume or invent a number, a rule or a date. Every number and date you use must come from an Observation.
- Before using date_math you must have searched the documents for the number of days.
- Use the calculator only for real arithmetic, never to repeat a number.
- A contract term that ends on a date renews on the next day, so a term ending 2030-05-31 renews on 2030-06-01. The cancellation deadline is the renewal date minus the notice days.
- Never answer from your own knowledge, only from tool results.
- Rules like discounts, limits and conditions come from the documents, so search for them. Numbers inside the tables come from analyze_data.
- Write one action at a time."""

answer_prompt = """You write the final answer for a company assistant.
Use only the numbered evidence. Write one short sentence per fact, and put the evidence number like [1] at the end of every sentence.
Example: The plan is Business [3]. The monthly fee is 468 dollars [3].
Cite only the evidence that directly supports that sentence. Never cite evidence you did not use.
Do not use evidence that is not relevant to the question.
If two sources give different values, mention both and say when each one applies.
Answer only what the question asks. If a part that the question asks for is missing from the evidence, say which part is missing. Never mention anything else that the documents do not say.
Give dates as full dates, for example January 25, 2027.
Only if nothing in the evidence helps at all, reply exactly: I could not find this information in the company documents.
Keep the answer short and clear."""

not_found = "I could not find this information in the company documents."

def make_plan(question):
    messages = [{"role": "system", "content": plan_prompt}, {"role": "user", "content": question}]
    plan = ask_llm(messages, 200)
    if len(plan) < 10:
        plan = "1. " + question
    return plan

def read_action(reply):
    match = re.search(r"Action:\s*(\w+)\s*\n\s*Action Input:\s*(.+)", reply, re.S)
    if match is None:
        return None, None
    return match.group(1).strip(), match.group(2).strip().strip('"')

def use_tool(action, action_input):
    if action not in tools:
        return "Unknown tool. Use one of: search_docs, summarize, analyze_data, calculator, date_math."
    return tools[action](action_input)

def write_answer(question):
    if len(evidence) == 0:
        search_docs(question)
    if len(evidence) == 0:
        return not_found, []
    numbered = "\n\n".join("[" + str(i + 1) + "] " + e["text"] for i, e in enumerate(evidence))
    messages = [{"role": "system", "content": answer_prompt}, {"role": "user", "content": "Question: " + question + "\n\nEvidence:\n" + numbered}]
    answer = ask_llm(messages, 500)
    if answer.startswith("I could not find"):
        return not_found, []
    cited = []
    for group in re.findall(r"\[([\d,\s]+)\]", answer):
        for n in re.findall(r"\d+", group):
            cited.append(int(n))
    cited = sorted(set(cited))
    cited = [n for n in cited if 1 <= n <= len(evidence)]
    if len(cited) == 0:
        cited = list(range(1, len(evidence) + 1))
    sources = ["[" + str(n) + "] " + evidence[n - 1]["label"] for n in cited]
    return answer, sources

def run_agent(question):
    evidence.clear()
    trace = []
    plan = make_plan(question)
    trace.append("Plan:\n" + plan)
    messages = [{"role": "system", "content": agent_prompt}, {"role": "user", "content": "Question: " + question + "\n\nPlan:\n" + plan}]
    for step in range(max_steps):
        reply = ask_llm(messages, 300).split("Observation:")[0].strip()
        trace.append("Step " + str(step + 1) + ":\n" + reply)
        if "Final Answer" in reply:
            break
        action, action_input = read_action(reply)
        messages.append({"role": "assistant", "content": reply})
        if action is None:
            messages.append({"role": "user", "content": "Use the exact format: Thought, Action, Action Input."})
            continue
        observation = use_tool(action, action_input)
        trace.append("Observation:\n" + observation[:600])
        messages.append({"role": "user", "content": "Observation: " + observation})
    answer, sources = write_answer(question)
    return answer, sources, "\n\n".join(trace)

answer, sources, trace = run_agent("What are Delta Logistics' plan, number of agents and monthly fee, and by what date must they cancel to avoid renewal?")
print(trace)
print()
print(answer)
print(sources)

Plan:
1. Find Delta Logistics' plan, number of agents, and monthly fee in the sales table
2. Search the contract for the term end date
3. Search the documents for the cancellation notice days
4. Calculate the cancellation date

Step 1:
Thought: I need to find Delta Logistics' plan, number of agents, and monthly fee
Action: search_docs
Action Input: Delta Logistics plan number agents monthly fee

Observation:
Document: Sales_data.csv | Section: row 3 (2026-02-01)
Date: 2026-02-01, Customer: Delta Logistics, Plan: Business, Agents: 12, Billing: Monthly, Amount USD: 468.0, Region: Egypt, Sales rep: Karim Fathy

Document: Incident_report.txt | Section: Affected customers
- Delta Logistics (Business plan, 12 agents, billed monthly): web app and live chat unavailable.
- Nile Bank (Enterprise plan, 25 agents, billed monthly): web app and API unavailable. Opened a P1 ticket at 09:20 and received the first response at 09:55.
- Cairo Bites (Starter plan, 3 agents, billed monthly): web app unavai

## 10. Test questions

In [14]:
test_questions = ["How many days of annual leave does a full-time employee get?", "If I lose my company laptop, who do I call and how fast?", "What is the yearly cost of 10 agents on the Business plan with annual billing?", "Which plan generated the most revenue in Q3 2026?", "Which employees are entitled to 30 days of annual leave?", "Is Hossam Ibrahim still on probation, and can he take 5 days of leave?", "Which customers are entitled to a service credit for the October incident, and how much is each?", "Can I work from home?", "What is the CEO's name?"]

for q in test_questions:
    answer, sources, trace = run_agent(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:", sources)
    print("-" * 70)

Q: How many days of annual leave does a full-time employee get?
A: [6] Full-time employees receive 21 working days of paid annual leave per year.
Sources: ['[6] HR_policy.pdf / 3. Annual leave']
----------------------------------------------------------------------
Q: If I lose my company laptop, who do I call and how fast?
A: If you lose your company laptop, you should call the IT Security Desk on extension 4100 or report by email to security@novatech.example within 2 hours. Reporting later than 2 hours may lead to disciplinary action.
Sources: ['[1] IT_security_policy.txt / 3. Lost or stolen devices', '[2] Employee_handbook.docx / Equipment and badges']
----------------------------------------------------------------------
Q: What is the yearly cost of 10 agents on the Business plan with annual billing?
A: 4680 USD [5]
Sources: ['[5] Data tables']
----------------------------------------------------------------------
Q: Which plan generated the most revenue in Q3 2026?
A: The Busines

## 11. Gradio app

In [35]:
def answer_question(question):
    if question.strip() == "":
        return "", "", ""
    answer, sources, trace = run_agent(question)
    return answer, "\n".join(sources), trace

with gr.Blocks() as demo:
    gr.Markdown("# Agentic RAG Assistant")
    gr.Markdown("Ask a question about the company documents.")
    with gr.Row():
        with gr.Column():
            question_box = gr.Textbox(label="Your question", lines=3)
            with gr.Row():
                clear_button = gr.Button("Clear")
                submit_button = gr.Button("Submit", variant="primary")
        with gr.Column():
            answer_box = gr.Textbox(label="Answer", lines=5)
            sources_box = gr.Textbox(label="Sources", lines=4)
            process_box = gr.Textbox(label="Agent process", lines=12)
    submit_button.click(answer_question, inputs=question_box, outputs=[answer_box, sources_box, process_box])
    clear_button.click(lambda: ("", "", "", ""), outputs=[question_box, answer_box, sources_box, process_box])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://7266c49de762ab3772.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
